# Task 2: Short-Term Stock Price Prediction

## Objective
This notebook implements a machine learning model to predict the next day's closing stock price using historical data from Yahoo Finance. We'll use features like Open, High, Low, and Volume to predict the next day's Close price.

## Libraries and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import sklearn
import warnings
warnings.filterwarnings('ignore')

# Show library versions
print("Library Versions:")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"YFinance version: {yf.__version__}")

# Create figures directory if it doesn't exist
import os
os.makedirs('./figures', exist_ok=True)
print("\nFigures directory created successfully.")

: 

## Data Collection

In [ ]:
# Fetch stock data from Yahoo Finance
ticker = 'AAPL'  # Using Apple as the stock ticker
print(f"Fetching data for {ticker}...")

# Download at least 3 years of daily data
stock_data = yf.download(ticker, period="3y", interval="1d")

print(f"Data downloaded successfully!")
print(f"Dataset shape: {stock_data.shape}")
print(f"Date range: {stock_data.index.min()} to {stock_data.index.max()}")

## Data Processing

In [ ]:
# Display basic information about the dataset
print("Dataset Info:")
print(stock_data.info())

print("\nFirst 5 rows:")
print(stock_data.head())

print("\nStatistical Summary:")
print(stock_data.describe())

# Check for missing values
missing_values = stock_data.isnull().sum()
print(f"\nMissing values per column:\n{missing_values}")

## Feature Engineering

In [ ]:
# Create features and target
# Features: Open, High, Low, Volume
features = ['Open', 'High', 'Low', 'Volume']
X = stock_data[features].copy()

# Target: Next day's Close price (shifted by -1 to get tomorrow's close)
y = stock_data['Close'].shift(-1)

# Combine X and y to drop rows with NaN values
df = pd.concat([X, y], axis=1)
df.columns = features + ['Target']

# Drop rows with NaN (last row will have NaN target)
df = df.dropna()

print(f"Shape after feature engineering: {df.shape}")
print(f"Features: {features}")
print(f"Target: Next day's Close price")

X = df[features]
y = df['Target']

print(f"Final feature matrix shape: {X.shape}")
print(f"Final target vector shape: {y.shape}")

## Train-Test Split

In [ ]:
# Split the data chronologically (no shuffle to maintain time series nature)
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Training date range: {stock_data.index[:split_index][0]} to {stock_data.index[:split_index][-1]}")
print(f"Testing date range: {stock_data.index[split_index:][0]} to {stock_data.index[split_index:][-1]}")

## Model Training

In [ ]:
# Train Linear Regression model
print("Training Linear Regression model...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test)

# Calculate metrics
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print(f"Linear Regression - MAE: {lr_mae:.2f}")
print(f"Linear Regression - RMSE: {lr_rmse:.2f}")

# Also train a Random Forest model for comparison
print("\nTraining Random Forest model...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# Calculate metrics
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"Random Forest - MAE: {rf_mae:.2f}")
print(f"Random Forest - RMSE: {rf_rmse:.2f}")

## Model Comparison and Selection

In [ ]:
# Compare models and select the better one
if lr_mae <= rf_mae:
    best_model = lr_model
    best_predictions = y_pred_lr
    best_model_name = "Linear Regression"
    print(f"Linear Regression performs better (lower MAE: {lr_mae:.2f} vs {rf_mae:.2f})")
else:
    best_model = rf_model
    best_predictions = y_pred_rf
    best_model_name = "Random Forest"
    print(f"Random Forest performs better (lower MAE: {rf_mae:.2f} vs {lr_mae:.2f})")

print(f"Selected model: {best_model_name}")

# Use the best model for final metrics
final_mae = mean_absolute_error(y_test, best_predictions)
final_rmse = np.sqrt(mean_squared_error(y_test, best_predictions))

print(f"Final model MAE: {final_mae:.2f}")
print(f"Final model RMSE: {final_rmse:.2f}")

## Visualization

In [ ]:
# Plot Actual vs Predicted Close Prices
plt.figure(figsize=(12, 6))
plt.plot(range(len(y_test)), y_test.values, label='Actual Close Price', alpha=0.7)
plt.plot(range(len(best_predictions)), best_predictions, label='Predicted Close Price', alpha=0.7)
plt.title(f'Actual vs Predicted Close Prices - {best_model_name} Model')
plt.xlabel('Time Index')
plt.ylabel('Close Price ($USD)')
plt.legend()
plt.grid(True, alpha=0.3)

# Save the plot
plt.tight_layout()
plot_filename = './figures/task2_stock_prediction.png'
plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
plt.show()

print(f"Visualization saved to: {plot_filename}")

## Model Performance Summary

In [ ]:
# Print detailed performance metrics
print("## Model Performance Summary")
print(f"Best Model: {best_model_name}")
print(f"Mean Absolute Error (MAE): ${final_mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${final_rmse:.2f}")
print(f"Average Close Price in Test Set: ${y_test.mean():.2f}")

# Calculate percentage error relative to average price
mae_percentage = (final_mae / y_test.mean()) * 100
print(f"MAE as Percentage of Average Price: {mae_percentage:.2f}%")

## Feature Importance (for Random Forest) or Coefficients (for Linear Regression)

In [ ]:
if best_model_name == "Random Forest":
    # Feature importance for Random Forest
    feature_importance = best_model.feature_importances_
    importance_df = pd.DataFrame({
        'Feature': features,
        'Importance': feature_importance
    }).sort_values('Importance', ascending=False)
    
    print("Feature Importance (Random Forest):")
    print(importance_df)
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x='Importance', y='Feature')
    plt.title('Feature Importance - Random Forest Model')
    plt.xlabel('Importance')
    plt.tight_layout()
    importance_plot_filename = './figures/task2_feature_importance.png'
    plt.savefig(importance_plot_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Feature importance plot saved to: {importance_plot_filename}")
    
else:  # Linear Regression
    # Feature coefficients for Linear Regression
    coefficients = best_model.coef_
    coef_df = pd.DataFrame({
        'Feature': features,
        'Coefficient': coefficients
    })
    
    print("Feature Coefficients (Linear Regression):")
    print(coef_df)
    
    # Plot coefficients
    plt.figure(figsize=(10, 6))
    bars = sns.barplot(data=coef_df, x='Coefficient', y='Feature')
    plt.title('Feature Coefficients - Linear Regression Model')
    plt.xlabel('Coefficient Value')
    plt.axvline(x=0, color='red', linestyle='--', alpha=0.5)
    plt.tight_layout()
    coef_plot_filename = './figures/task2_coefficients.png'
    plt.savefig(coef_plot_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Coefficients plot saved to: {coef_plot_filename}")

## Observations & Insights

In [ ]:
if best_model_name == "Random Forest":
    feature_analysis = features[np.argmax(feature_importance)]
    method = "feature importance"
else:
    feature_analysis = features[np.argmax(np.abs(coefficients))]
    method = "coefficient magnitude"

observations = f"""
## Observations & Insights

Based on the stock price prediction model, here are the key findings:

1. **Model Performance**: The model achieved a Mean Absolute Error (MAE) of approximately ${final_mae:.2f}, which represents {mae_percentage:.2f}% of the average stock price in the test set. This indicates reasonable predictive accuracy for short-term stock price forecasting.

2. **Feature Impact**: Among the input features (Open, High, Low, Volume), {feature_analysis} appears to be the most influential predictor for the next day's closing price, based on the {method}.

3. **Prediction Accuracy**: The model shows ability to capture general trends in the stock price movement, though predicting exact prices remains challenging due to the inherent volatility of financial markets.

4. **Time Series Nature**: Using a chronological train-test split preserved the temporal nature of the data, which is crucial for financial forecasting.

5. **Limitations**: Stock prices are influenced by numerous external factors (news, market sentiment, economic indicators) not captured in historical price/volume data alone.
"""

print(observations)

## Next Steps

In [ ]:
next_steps = """
## Next Steps

Based on this exploratory analysis, here are suggested next steps:

- **Advanced Models**: Experiment with LSTM or other time-series specific models for improved accuracy
- **Technical Indicators**: Include technical indicators (RSI, MACD, moving averages) as additional features
- **External Data**: Incorporate market sentiment, news data, or economic indicators
- **Cross-Validation**: Implement time series cross-validation for more robust model evaluation
- **Risk Assessment**: Develop confidence intervals for predictions to quantify uncertainty
"""

print(next_steps)

## Submission Checklist

In [ ]:
checklist = """
## Submission Checklist

- [x] Imported required libraries (pandas, numpy, matplotlib, seaborn, sklearn, yfinance)
- [x] Fetched stock data (AAPL) with at least 2-5 years of daily data
- [x] Performed data processing (.head(), .info(), .describe())
- [x] Created target column by shifting Close price by -1 day
- [x] Dropped rows with NaN target values
- [x] Defined features (Open, High, Low, Volume) and target (Next day Close)
- [x] Applied chronological train-test split (80/20)
- [x] Trained Linear Regression or Random Forest model
- [x] Computed MAE and RMSE metrics
- [x] Created Actual vs Predicted visualization
- [x] Saved visualization to ./figures/
- [x] Added model performance summary
- [x] Included feature importance/coefficient analysis
- [x] Added observations and insights section
- [x] Included next steps suggestions
"""

print(checklist)

print("\nTask 2: Short-Term Stock Price Prediction - COMPLETED")
print("All required deliverables have been created and saved.")